## Climate Change Impact Prediction Using a Power Law Correlation


This covers the **use phase and end-of-life phase**, where the impact comes from the process technology such as distillation, pervaporation, drying, or membrane filtration, rather than the chemical itself. It estimates a technology's GWP by scaling from a known reference case, using two operating parameters: throughput (F) and energy consumption (E).

$$\frac{GWP_{new}}{GWP_{ref}} = \left(\frac{F_{new}}{F_{ref}}\right)^{\alpha}\left(\frac{E_{new}}{E_{ref}}\right)^{\beta}$$

α and β show how sensitive the GWP is to throughput and energy

### Estimating α and β for dryer
This notebook estimates α and β for the dryer from reference F, E, and GWP data.

### 1. Import packages for the fit
**numpy:** does fast calculations on the F, E, and GWP arrays

**scipy.optimize (minimize):** finds the α and β that best match the data by minimising the error

**matplotlib.pyplot:** draws the parity plot of actual vs predicted GWP

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

### 2. Dryer reference data
Each row is a dryer case with its throughput (F) and energy (E) in `Param`, and its GWP in `GWPact`.

`Fref`, `Eref`, and `GWPref` are the reference case, the highest-throughput point that everything is scaled against.

In [ ]:
GWPact = np.array([
    8.84E+02,
    1.52E+02,
    1.29E+02,
    1.27E+02,
    1.18E+00,
    3.69E+02,
    1.95E+03
])

Param = np.array([
    [0.00104,      1.47E+02],
    [0.01,         1.00E+01],
    [0.02,         8.33E+00],
    [0.04,         1.57E+01],
    [0.001079137,  5.25E+00],
    [0.307246377,  1.60E+03],
    [1.645410628,  8.47E+03]
])

Fref = 1.645410628
Eref = 8.47E+03
GWPref = 1.95E+03

### 3. Determine α and β
Normalises F and E against the reference case, then searches for the α and β that make the predicted GWP match the actual GWP as closely as possible

In [ ]:
X1 = Param[:, 0] / Fref
X2 = Param[:, 1] / Eref

def residuals(k):
    GWPpred = (X1**k[0] * X2**k[1]) * GWPref
    return np.sum((np.log(GWPact) - np.log(GWPpred))**2)

x0 = [0.07, 0.49]
res = minimize(residuals, x0, method='Nelder-Mead')
alpha, beta = res.x

### Show the results
Prints the fitted α and β, and the R² score showing how well the equation matches the actual GWP (1.0 is a perfect fit)

In [ ]:
GWPpred = (X1**alpha * X2**beta) * GWPref

ss_res = np.sum((GWPact - GWPpred)**2)
ss_tot = np.sum((GWPact - np.mean(GWPact))**2)
r2 = 1 - ss_res / ss_tot

print("alpha =", round(alpha, 4))
print("beta  =", round(beta, 4))
print("R2 score    =", round(r2, 4))

### Parity plot: actual vs predicted GWP
Plots the actual GWP against the predicted GWP for each case. Points on the dotted
diagonal are perfectly predicted; the further a point sits from the line, the larger
its prediction error.

In [ ]:
# Predicted GWP from the fitted alpha, beta
GWPpred = (X1**alpha * X2**beta) * GWPref

fig, ax = plt.subplots(figsize=(6, 5))

# Orange points
ax.scatter(GWPact, GWPpred, color='#E8703A', s=45, zorder=3)

# Dotted trend line through origin (y = x reference)
line_max = 2500
ax.plot([0, line_max], [0, line_max], color='#E8703A',
        linestyle=':', linewidth=1.5, zorder=2)

# Axes: linear, 0 to 2500, equal ticks
ax.set_xlim(0, 2500)
ax.set_ylim(0, 2500)
ax.set_xticks(range(0, 2501, 500))
ax.set_yticks(range(0, 2501, 500))

# R2 annotation
ax.text(300, 2000, f"R² = {round(r2, 4)}", fontsize=12)

ax.set_xlabel("Actual GWP kgco$_2$-eq/kg$_{chem}$", fontweight='bold')
ax.set_ylabel("Predicted GWP kgco$_2$-eq/kg$_{chem}$", fontweight='bold')
ax.set_title("Dryer", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()